# TMDWF CS-Kernel Averaging Template

This notebook is a self-contained wrapper around the repository's downstream TMDWF CS-kernel averaging workflow.
It reads already-generated CS-kernel outputs, selects x values using the momentum and bT thresholds or an explicit x range, writes averaged values with bootstrap statistical and systematic errors, and automatically generates a bT summary plot.


## Imports / Setup

Run this notebook from the repository root, or adjust `REPO_ROOT` below.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from lqcd_analysis.notebook_workflows import (
    pretty_print_config,
    run_tmdwf_cs_kernel_average_from_notebook,
    validate_tmdwf_cs_kernel_average_notebook_config,
)


## User Inputs

## Data settings

This workflow consumes existing CS-kernel outputs and averages the x-dependent results over the selected x window for each bT.

The selection uses the repository's built-in cuts:

- `2 * x * pz > 1 GeV`
- `2 * (1 - x) * pz > 1 GeV`
- `bT * pz * x > 0.5`
- `bT * pz * (1 - x) > 0.5`


In [ ]:
workflow_config = {
    # Data settings
    "title_pattern": "l64c64a076_m140_fit_pz*",
    "input_root": REPO_ROOT / "5-CSkernel",
    "lattice_spacing_fm": 0.076,
    "gm": "T5",
    "eta": "eta0",
    "component": "real",
    "nstates": 1,
    "normalization_mode": "raw",
    "reference_pz_labels": ["5-6", "6-7", "7-8"],

    # CS-kernel averaging settings
    "scheme": "CG",
    "extraction_type": "type2",
    "kernel_label": "LO",
    "bTrange": [0, 20],
    "x_range": [0.25, 0.75],

    # Output settings
    "results_dir": REPO_ROOT / "6-CSkernel-average",
}


## Option Guide

Edit only `workflow_config` in the cell above for normal usage.
This notebook does not depend on a separate input file; the `workflow_config` cell is the source of truth for notebook runs.

- `input_root`: Root directory containing existing TMDWF CS-kernel outputs. The workflow resolves the usual CS-kernel table/sample filenames automatically from the repository naming convention.
- `lattice_spacing_fm`: Lattice spacing in fm used to convert `bT` to `bT[fm]` for the summary plot and tabular output.
- `title_pattern`: Same per-`pz` title pattern used upstream by the TMDWF fit/Fourier/CS-kernel workflows, for example `l64c64a076_m140_fit_pz*`.
- `gm`, `eta`: Operator and insertion-channel selectors used to identify the correct CS-kernel outputs.
- `component`, `nstates`: Select which Fourier/CS-kernel family to consume.
- `normalization_mode`: One of `raw`, `mode1`, `mode2`, or `mode3`. This must match the upstream Fourier/CS-kernel output mode.
- `scheme`: Matching scheme selector. The current repository implementation supports only `"CG"` for the type-2 TMDWF workflow, and will raise a clear validation error otherwise.
- `extraction_type`: Keep this at `"type2"` for the legacy qTMDWF CS-kernel method.
- `kernel_label`: Perturbative label to average. The current workflow expects one label at a time, for example `LO` or `NLL`.
- `bTlist` or `bTrange`: Which transverse separations to process. The workflow loops over every requested `bT`.
- `x_range`: Optional explicit `x_min, x_max` interval. When provided, the workflow uses this x range directly and skips the physical x-selection cuts.
- `reference_pz_labels`: Which CS-kernel pair groups to include. If omitted, all matching pair-group labels are used.
- `results_dir`: Output root for the averaged summary file, grouped tables, bootstrap samples, and selection metadata.

Expected input data shape:

- The workflow reads repository-native CS-kernel band tables and bootstrap sample tables.
- It groups the CS-kernel x-dependent results by `bT`, applies the physical x-selection cuts, and then averages across the selected x values for each bootstrap sample.
- The output summary uses the mean of the sample means as the central value, a percentile-based statistical error from the sample means, and the mean within-sample standard deviation as the systematic error.

What the workflow writes:

- `*_summary.txt`: metadata/provenance snapshot
- `tables/*_values.txt`: one averaged value and error budget per `bT`
- `tables/*_selection.txt`: the x-selection summary used for each source CS-kernel file
- `samples/*_samples.txt`: bootstrap averages and within-sample spreads for each `bT`
- `plots/*_bT_average.pdf`: automatic `bT` summary plot with total and statistical error bars


## Validate / Preview


In [ ]:
validated_config = validate_tmdwf_cs_kernel_average_notebook_config(workflow_config)
print(pretty_print_config(validated_config))



## Run Backend Workflow

Uncomment the cell below to execute the averaging workflow and write outputs under `results_dir`.


In [ ]:
# outputs = run_tmdwf_cs_kernel_average_from_notebook(workflow_config)
# for output in outputs:
#     print(output)
